In [ ]:
from proto import *
from engine import *
from utils import *
from runners import CompartmentalModel

# from experimental import *
from managed import *

import pandas as pd
import numpy as np
import datetime as dt

pd.options.plotting.backend = "plotly"

In [ ]:
import computegraph as cg

In [ ]:
class Parameter(cg.Variable):
    def __init__(self, key, default):
        super().__init__(key, "parameters")
        self.default = default


class ModelVariable(cg.Variable):
    def __init__(self, key):
        super().__init__(key, "model_variables")
        self.key = key


Time = ModelVariable("time")
CompartmentValues = ModelVariable("compartment_values")
param = Parameter


def defer(func, name=None):
    def _proxy(*args, **kwargs):
        return cg.Function(func, args, kwargs, name)

    return _proxy


def label(graph_obj, name):
    graph_obj.node_name = name
    return graph_obj

In [ ]:
from utils import LinearInterpolator

In [ ]:
from computegraph.types import Data

In [ ]:
epoch = Epoch(dt.datetime(1980, 6, 7))

In [ ]:
x = jnp.array([0.0, 15.0, 20.0, 40.0])
y = jnp.array([0.0, 0.0, 2.0, 0.0])

t = LinearInterpolator(x, y)

t.process(jnp.linspace(0.0, 50.0))

In [ ]:
double_time = defer(lambda t: t * 2.0, "doubletime")(Time * 2.0)
vinterp = defer(LinearInterpolator, "CRInterpolator")(
    Data(x, "cr_times"), Data(y, "cr_values")
)

In [ ]:
vacct = defer(LinearInterpolator.process)(vinterp, Time)

In [ ]:
disease_state = Stratification("disease_state", ["S", "I", "R"])
humans = CompartmentMap.new(disease_state)
humans.compartments

In [ ]:
age_strat = humans.stratify(Stratification("age", ["child", "adult"]))

In [ ]:
loc_strat = humans.stratify(Stratification("location", ["N", "S", "E", "W"]))

In [ ]:
class InfectionProcess:
    def __init__(
        self,
        mm: ManagedArray,
        infectee_cats: CategoryGroup,
        infector_cats: CategoryGroup,
        disease_strat: Stratification,
    ):
        self.mm = mm
        self.infector_cats = infector_cats
        self.infectee_cats = infectee_cats
        self.disease_strat = disease_strat
        self._infectious_pop_cats = self.infector_cats.product(disease_strat["I"])

    def process(self, compartment_values: ManagedArray, contact_rate: float):
        ipops = compartment_values.sumcats(self._infectious_pop_cats)
        total_pop = compartment_values.sumcats(self.infector_cats)
        age_foi = self.mm.data @ (ipops.data / total_pop.data) * contact_rate
        return CategoryData(self.infectee_cats, age_foi)

In [ ]:
mmnp = np.array([[1.8, 0.3], [0.2, 0.4]])

spatial_mm = np.random.normal(1.0, 0.1, (4, 4))

jnp.kron(mmnp, spatial_mm)

mm = ManagedArray(mmnp, ["dest", "source"])
mm.indices["source"] = ManagedCategoryGroupIndex("source", age_strat.categories())
mm.indices["dest"] = ManagedCategoryGroupIndex("dest", age_strat.categories())

infectees = age_strat.categories()  # .product(disease_state["S"])
infectors = age_strat.categories()  # .product(disease_state["I"])

iprocess = defer(InfectionProcess)(mm, infectees, infectors, disease_state)

In [ ]:
foi = defer(InfectionProcess.process)(
    iprocess, CompartmentValues, Parameter("contact_rate", 0.2)
)

In [ ]:
infection = TransitionFlow(disease_state["S"], disease_state["I"], foi)
recovery = TransitionFlow(
    disease_state["I"], disease_state["R"], Parameter("recovery_rate", 0.1)
)

In [ ]:
adj_arr = vacct * np.array([0.0, 1.0, 0.5, 1.1])

In [ ]:
infection.adjustments.append(
    defer(CategoryData, "adj_spatial_infection")(
        loc_strat.categories(),
        adj_arr,
    )
)

In [ ]:
epoch

In [ ]:
flows = {"infection": infection, "recovery": recovery}
model = CompartmentalModel(humans, flows)

runner = model.get_runner(50, epoch)

runner.graph.draw()

In [ ]:
istate = humans.zeros(np).data
S_idx = humans.query(disease_state["S"]).indices
istate[S_idx] = 100.0
istate[humans.query(age_strat["child"]).indices] *= 0.5
istate[humans.query([loc_strat["N"], disease_state["I"]]).indices] = 1.0
istate[humans.query([loc_strat["W"], disease_state["I"]]).indices] = 2.0
params = {"contact_rate": 0.4, "recovery_rate": 0.1}

results = runner.run(istate, params)

In [ ]:
compres = results["compartments"]
flowres = results["flows"]

In [ ]:
compres.to_pandas_df().plot()

In [ ]:
infdata = flowres["infection"]

In [ ]:
infdata.sumcats(source=loc_strat.categories()).to_pandas_df().plot()